# Minería de Datos — Sesión 2
### Universidad Distrital · 26160 · grupo 020‑81 · jueves 13 de agosto de 2026

**Objetivo de este notebook:** recorrer **CRISP‑DM completo** —las seis fases— sobre un caso pequeño,
en una sola sesión, para que el mapa quede visto de punta a punta antes de empezar a bajar al detalle.

De aquí en adelante cada semana profundiza **una** de estas fases. Hoy se ve el recorrido entero,
aunque cada pieza quede corta a propósito.

> **Cómo se trabaja este notebook:** el docente escribe en vivo; usted ejecuta en el suyo. Al final
> se comparte esta versión resuelta. No copie mientras se explica — mire, y después ejecute.

---
## 0. Python en doce minutos *(opcional)*

Si nunca ha programado en Python, ejecute esta sección. Si ya programa, sáltela: no se pierde nada
del resto del notebook.

In [ ]:
# Una lista: colección ordenada, se indexa desde 0
notas = [3.2, 4.5, 2.8, 3.9]
print(notas[0], notas[-1], len(notas))

# Un diccionario: pares clave -> valor
estudiante = {"codigo": "20211020148", "programa": "Sistemas", "promedio": 3.9}
print(estudiante["programa"])

# Un ciclo y una condición
for n in notas:
    print(n, "aprueba" if n >= 3.0 else "pierde")

Un `DataFrame` de pandas es, en la práctica, **un diccionario de listas con nombre**: cada clave es
una columna y cada lista es su contenido. Todo lo que sigue son operaciones sobre esa estructura.

In [ ]:
import pandas as pd

mini = pd.DataFrame({"nombre": ["Ana", "Luis", "Sara"],
                     "nota":   [4.1, 2.7, 3.5]})

print(mini["nota"].mean())      # una columna, y su promedio
print(mini[mini["nota"] >= 3])  # filtrar filas por condición

---
## 1. Fase 1 · Comprensión del negocio

**El caso.** Bienestar Institucional quiere reducir la deserción en los programas de Ingeniería.
Hoy se enteran del retiro cuando ya ocurrió: el estudiante simplemente no vuelve a matricularse.
Tienen presupuesto para acompañar a **150 estudiantes por semestre** —tutorías, apoyo económico,
seguimiento— y hoy eligen a quién acompañar sin criterio.

Esta fase **no se hace en el computador**. Se hace preguntando. Y produce cuatro cosas escritas:

In [ ]:
ficha = {
    "pregunta_de_negocio":
        "¿A cuáles 150 estudiantes acompañamos este semestre para que no se retiren?",

    "pregunta_de_mineria":
        "Dado el historial académico y socioeconómico al cierre de un semestre, "
        "¿se puede estimar la probabilidad de que el estudiante NO se matricule en el siguiente?",

    "criterio_de_exito":
        "Que entre los 150 señalados haya más retiros reales que en una selección al azar "
        "(hoy: ~25 % de la población se retira; el azar acertaría en unos 37 de los 150).",

    "restricciones":
        "No se puede usar información posterior al momento de la decisión. "
        "Sin datos personales identificables en el informe.",
}

for k, v in ficha.items():
    print(f"{k}:\n  {v}\n")

> **La distinción que hay que llevarse de esta fase.** «Reducir la deserción» es un *objetivo de
> negocio*, no una tarea de minería. «Estimar la probabilidad de no matrícula» sí lo es: tiene una
> unidad de análisis (el estudiante), un momento de corte y una respuesta verificable.
> Un proyecto que no logra hacer esa traducción **no falla en el modelado: ya falló aquí**.

**El criterio de éxito se escribe hoy, no en noviembre.** Si se define después de ver los
resultados, siempre se cumple.

---
## 2. Fase 2 · Comprensión de los datos

Registro Académico entrega un archivo con los estudiantes matriculados el semestre pasado.

La celda siguiente **simula** ese archivo para que el notebook corra sin depender de nada externo.
En su proyecto esta celda es un `pd.read_csv(...)` y nada más.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(2026)
n = 1200

programa   = rng.choice(["Sistemas", "Industrial", "Catastral", "Electrónica"], n, p=[.34, .26, .22, .18])
jornada_ok = rng.choice(["Diurna", "Nocturna"], n, p=[.55, .45])
edad       = rng.integers(17, 33, n)
estrato    = rng.choice([1, 2, 3, 4], n, p=[.18, .42, .32, .08])
promedio   = np.clip(rng.normal(3.5, 0.45, n), 1.5, 5.0).round(2)
creditos   = rng.integers(9, 21, n)
repeticion = rng.choice([0, 1, 2], n, p=[.72, .22, .06])
apoyo      = rng.choice(["Sí", "No"], n, p=[.30, .70])
distancia  = np.clip(rng.gamma(2.0, 4.0, n), 0.5, 45).round(1)

# el retiro depende de verdad de algunas de esas variables
z = (-1.75
     - 1.60 * (promedio - 3.5)
     + 0.55 * repeticion
     + 0.030 * distancia
     - 0.40 * (apoyo == "Sí")
     + 0.35 * (estrato <= 2))
retiro = rng.random(n) < 1 / (1 + np.exp(-z))

# y una columna que Registro incluyó "por si sirve"
matricula_sig = np.where(retiro,
                         rng.choice(["Sin pagar", "Pagada"], n, p=[.95, .05]),
                         rng.choice(["Sin pagar", "Pagada"], n, p=[.03, .97]))

# --- la suciedad de siempre: así llega un archivo real ---
variantes = {"Diurna": ["Diurna", "DIURNA", "diurna  "],
             "Nocturna": ["Nocturna", "NOCTURNA", "nocturna "]}
jornada = [rng.choice(variantes[j]) for j in jornada_ok]

promedio_txt = []
for v, u in zip(promedio, rng.random(n)):
    if u < 0.06:
        promedio_txt.append("N/A")          # faltante disfrazado de texto
    elif u < 0.09:
        promedio_txt.append("-99")          # faltante disfrazado de número
    else:
        promedio_txt.append(f"{v:.2f}".replace(".", ","))   # coma decimal

distancia = distancia.astype(float)
distancia[rng.random(n) < 0.04] = np.nan    # faltante de verdad

base = pd.DataFrame({
    "id":                  np.arange(20260001, 20260001 + n),
    "programa":            programa,
    "jornada":             jornada,
    "edad":                edad,
    "estrato":             estrato,
    "promedio_anterior":   promedio_txt,
    "creditos_inscritos":  creditos,
    "asignaturas_repetidas": repeticion,
    "apoyo_financiero":    apoyo,
    "distancia_km":        distancia,
    "matricula_siguiente": matricula_sig,
    "retiro":              np.where(retiro, "Sí", "No"),
})

# quince filas repetidas, como cuando se pega el reporte de dos sedes
crudo = (pd.concat([base, base.sample(15, random_state=7)], ignore_index=True)
           .sample(frac=1, random_state=7)
           .reset_index(drop=True))

crudo.head()

### Las cuatro preguntas de la fase 2

Las mismas de la sesión 1, en el mismo orden. Hasta diciembre.

In [ ]:
print("forma:", crudo.shape)
print()
print(crudo.dtypes)

In [ ]:
crudo.isna().sum()

In [ ]:
crudo.describe()

**Pare y lea lo que dice ese resultado.** `isna()` reporta faltantes **solo en `distancia_km`** —y es
mentira: `promedio_anterior` tiene un 9 % de faltantes escritos como `"N/A"` y `"-99"`. Y `describe()`
no resume el promedio académico, que es justo la variable de la que sospechamos, porque pandas la
leyó como texto.

Antes de tocar nada: **mirar** las columnas sospechosas.

In [ ]:
print("jornada    :", crudo["jornada"].unique())
print("promedio   :", crudo["promedio_anterior"].unique()[:8], "...")
print("duplicados :", crudo.duplicated().sum())
print()
print(crudo["retiro"].value_counts(normalize=True).round(3))

> **La última línea es la más importante de la fase 2.** Se retira alrededor de **una cuarta parte**.
> Es decir: **un modelo que diga «nadie se retira» acierta las otras tres cuartas partes.** Ese
> número —la *tasa base*— es contra lo que hay que comparar cualquier resultado de hoy, no contra
> cero. Anótelo: vuelve en la fase 5.

---
## 3. Fase 3 · Preparación de los datos

Sobre una copia, siempre. Cuatro arreglos, uno por problema encontrado.

In [ ]:
d = crudo.drop_duplicates().copy()
print("filas:", len(crudo), "->", len(d))

# 1. promedio: coma decimal, "N/A" y "-99"
d["promedio_anterior"] = pd.to_numeric(
    d["promedio_anterior"].str.replace(",", ".", regex=False), errors="coerce")
d.loc[d["promedio_anterior"] < 0, "promedio_anterior"] = np.nan

# 2. jornada: espacios y mayúsculas
d["jornada"] = d["jornada"].str.strip().str.title()

print("faltantes reales de promedio:", d["promedio_anterior"].isna().sum())
print("jornada:", d["jornada"].unique())

### Imputar es una decisión, no un trámite

Hay tres salidas para un faltante: **borrar** la fila, **rellenar** con algo, o **dejarlo** y usar un
modelo que lo tolere. Hoy rellenamos con la mediana porque es rápido y hay que llegar a la fase 6.
En la semana 5 se ve el precio de esa decisión —y por qué a veces *el hecho de faltar* es la
información más valiosa de la columna.

In [ ]:
for col in ["promedio_anterior", "distancia_km"]:
    d[col] = d[col].fillna(d[col].median())

print(d.isna().sum().sum(), "faltantes en toda la tabla")

### Del cuadro de datos a la matriz que el modelo entiende

scikit‑learn no lee `"Sistemas"` ni `"Diurna"`: solo números. `get_dummies` convierte cada categoría
en columnas de 0/1.

In [ ]:
y = (d["retiro"] == "Sí").astype(int)
X = pd.get_dummies(d.drop(columns=["id", "retiro"]), drop_first=True)

print("X:", X.shape, " y:", y.shape)
print(list(X.columns))

In [ ]:
from sklearn.model_selection import train_test_split

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y)

print("entrenamiento:", X_tr.shape[0], " prueba:", X_te.shape[0])
print("proporción de retiro — entrenamiento:", round(y_tr.mean(), 3),
      "| prueba:", round(y_te.mean(), 3))

> **Por qué se parte en dos.** El modelo va a *memorizar* lo que se le muestre. Si lo evaluamos con
> los mismos datos con que aprendió, la nota es la de un examen con el cuaderno abierto. `stratify=y`
> mantiene el 25 % de retiros en ambos lados. Esto se ve en serio en la semana 6.

---
## 4. Fase 4 · Modelado

**Primero el modelo tonto.** Siempre. Es la vara contra la que se mide todo lo demás: predice
siempre la clase mayoritaria —«no se retira»— y no aprende nada.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeClassifier, export_text

tonto = DummyClassifier(strategy="most_frequent").fit(X_tr, y_tr)

arbol = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_tr, y_tr)

print(export_text(arbol, feature_names=list(X_tr.columns)))

---
## 5. Fase 5 · Evaluación

Dos preguntas, en este orden: **¿le gana al modelo tonto?** y **¿responde la pregunta de la fase 1?**

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix

for nombre, modelo in [("tonto", tonto), ("árbol", arbol)]:
    print(f"{nombre:6s} acierto = {accuracy_score(y_te, modelo.predict(X_te)):.3f}")

El árbol acierta casi siempre. **Y eso, en un problema de deserción, es una alarma, no un logro.**
Cuando un resultado es demasiado bueno, la fase 5 obliga a preguntar *de dónde salió*.

In [ ]:
importancia = (pd.Series(arbol.feature_importances_, index=X_tr.columns)
                 .sort_values(ascending=False))
importancia.head(5)

### Fuga de información

El modelo se apoya casi por completo en `matricula_siguiente`. Y esa columna **no existe en el
momento en que hay que tomar la decisión**: dice si el estudiante pagó la matrícula del semestre
siguiente, o sea, dice el retiro con otras palabras. Es la restricción que quedó escrita en la
fase 1 —«no se puede usar información posterior al momento de la decisión»— y la acabamos de violar.

Un modelo con fuga funciona perfecto en el notebook y **es inservible en producción**. Se vuelve a
la fase 3 y se saca la columna. Esto es el ciclo de CRISP‑DM, no un error de principiante: pasa en
proyectos reales, y lo caro es descubrirlo después de haberlo entregado.

In [ ]:
fuga = [c for c in X.columns if c.startswith("matricula_siguiente")]
X2 = X.drop(columns=fuga)

X2_tr, X2_te, y_tr, y_te = train_test_split(
    X2, y, test_size=0.30, random_state=42, stratify=y)

arbol2 = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X2_tr, y_tr)
tonto2 = DummyClassifier(strategy="most_frequent").fit(X2_tr, y_tr)

print("tonto :", round(accuracy_score(y_te, tonto2.predict(X2_te)), 3))
print("árbol :", round(accuracy_score(y_te, arbol2.predict(X2_te)), 3))
print()
print(export_text(arbol2, feature_names=list(X2_tr.columns)))

### La tabla que hay que mirar cuando el acierto no alcanza

In [ ]:
mc = confusion_matrix(y_te, arbol2.predict(X2_te))

print(pd.DataFrame(mc,
                   index=["real: se queda", "real: se retira"],
                   columns=["predice: se queda", "predice: se retira"]))

Léala por filas: de todos los que **sí se retiraron**, ¿a cuántos señaló el modelo? Ese número —y no
el acierto global— es el que responde la pregunta de la fase 1, porque el programa de Bienestar se
juega en a quién llama. Un modelo que acierta como el tonto y **casi no señala retiros es exactamente
el modelo tonto, con más pasos**.

Las métricas que ponen nombre a esto —precisión, exhaustividad, F1— son la **semana 7**. Hoy basta
con haber visto por qué el acierto solo no sirve.

> **Y con esto, ¿el proyecto se aprueba o se devuelve?** Con estos resultados, se devuelve a la
> fase 2: hacen falta variables del momento correcto —asistencia, notas parciales, historial de
> pagos anteriores—. Decidir eso **es** la fase 5. No es un anexo del modelado.

---
## 6. Fase 6 · Despliegue

Desplegar casi nunca es «montar el modelo en un servidor». Es **entregar la decisión** en el formato
en que quien decide la usa. Aquí: la lista de los 150 estudiantes que Bienestar va a llamar.

In [ ]:
prob = arbol2.predict_proba(X2_te)[:, 1]          # probabilidad estimada de retiro

lista = (d.loc[X2_te.index, ["id", "programa", "jornada", "promedio_anterior"]]
           .assign(prob_retiro=prob.round(3))
           .sort_values("prob_retiro", ascending=False)
           .head(10))

lista

Y con la lista van tres cosas que el modelo no dice y el informe sí tiene que decir:

- **Con qué datos se entrenó y de cuándo son.** Un modelo entrenado con 2025‑1 no vale para siempre.
- **Qué se sabe que le falta.** Aquí: no hay asistencia, no hay notas parciales, hay una columna que
  hubo que descartar por fuga.
- **Qué pasa si se equivoca.** Señalar de más cuesta una llamada; señalar de menos cuesta un
  estudiante que se va. No son costos iguales, y el umbral debería reflejarlo.

> **El despliegue devuelve al inicio.** Bienestar llama a 150 estudiantes, y en marzo se sabe a
> cuántos se retuvo. Ese resultado es la entrada de la siguiente vuelta de CRISP‑DM. Por eso es un
> ciclo dibujado en círculo y no una lista de seis pasos.

---
## 7. El recorrido completo, en una tabla

| Fase | Qué se hizo hoy | Semana en que se profundiza |
|---|---|---|
| 1 · Negocio | La ficha: pregunta, criterio de éxito, restricciones | hoy — y en su proyecto |
| 2 · Datos | `shape`, `dtypes`, `isna`, `describe`, `unique`, tasa base | 3 y 4 |
| 3 · Preparación | Duplicados, faltantes disfrazados, categorías, dummies | 5 |
| 4 · Modelado | Modelo tonto + árbol de profundidad 3 | 6 a 9 |
| 5 · Evaluación | Acierto vs. tasa base, fuga, matriz de confusión | 7 |
| 6 · Despliegue | La lista priorizada + los límites declarados | 17 y sustentación |

Del 100 % del tiempo de hoy, las fases 2 y 3 se llevaron la mitad —y este archivo estaba **casi
limpio y tenía 1 200 filas**. Ese es el 60‑80 % del que hablamos la sesión pasada.

---
## 8. Su turno — para hoy y para el sábado 22

**En clase (últimos 20 minutos), por grupo:** llenar la ficha de la fase 1 para su proyecto.
Las cuatro casillas del punto 1 de este notebook, con sus datos. Está en
`S02_Ficha_de_proyecto.html`, y de ahí sale el texto para pegar en el grupo.

**Con los dos candidatos que trajeron**, corran el diagnóstico de abajo sobre cada uno. La propuesta
formal se entrega el **sábado 22 de agosto**: ficha + este diagnóstico + las dos preguntas.

> Mínimos del conjunto de datos: **≥ 2 000 filas · ≥ 12 columnas · ≥ 6 numéricas continuas ·
> ≥ 3 categóricas · ≥ 1 fecha o texto · 1 objetivo categórico · 1 objetivo numérico · ≤ 200 MB**,
> y **con suciedad real**.

In [ ]:
from google.colab import files
subido = files.upload()

nombre = list(subido.keys())[0]
mio = pd.read_csv(nombre, sep=";", encoding="latin-1")   # ajuste sep y encoding si falla
mio.shape

In [ ]:
print("filas, columnas:", mio.shape)
print()
print("numéricas :", mio.select_dtypes(include="number").shape[1])
print("categóricas:", mio.select_dtypes(include="object").shape[1])
print()
print("faltantes por columna (las 10 peores):")
print(mio.isna().sum().sort_values(ascending=False).head(10))
print()
print("duplicados:", mio.duplicated().sum())

**Y la pregunta que decide si ese archivo sirve:** ¿cuál columna sería el objetivo categórico, y
cómo se reparte?

In [ ]:
objetivo = "CAMBIE_ESTO_POR_SU_COLUMNA"

mio[objetivo].value_counts(normalize=True).round(3)

Si la clase menor no llega al **10 %**, o si hay más de cinco clases con una sola dominando, ese
conjunto va a dar problemas en el bloque 3 —y el problema aparece en octubre, cuando ya no hay cómo
cambiarlo—. Tráiganlo a revisión antes del 22.

---

**Entrega de hoy:** ficha de la fase 1 del grupo, al grupo de WhatsApp, antes de las 10:00.
**Próxima sesión: sábado 15 de agosto, 8:00.** Fuentes, formatos y cómo se leen datos que no vienen
en un CSV limpio.